In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as T
import requests
from PIL import Image
from io import BytesIO

In [2]:
def download_image(url):
    '''
    Download an image from the given URL with a valid
    User Agent and return it as a PIL image
    '''
    headers={
        "User-Agent":(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status() # Raise an HTTPError if the request is unsuccessful
    return Image.open(BytesIO(response.content)).convert('RGB')

In [3]:
import requests

def load_imagenet_labels():
    """
    Download the 1,000 ImageNet class labels with a valid
    User-Agent and return them as a list indexed by class ID.
    """
    labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }
    
    response = requests.get(labels_url, headers=headers, verify=False)
    response.raise_for_status()
    labels_str = response.text.strip().split("\n")
    return labels_str

In [4]:
def preprocess_image(image):
    """
    Transform the PIL Image into a Tensor suitable for
    the pretrained PyTorch model.
    """
    transform = T.Compose([
        T.Resize(256),                                 # Resize the image (shortest side = 256)
        T.CenterCrop(224),                             # Crop the image at the center to 224x224
        T.ToTensor(),                                  # Convert the image to a PyTorch tensor
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])         # Normalize using ImageNet means and stds
    ])
    
    return transform(image).unsqueeze(0)               # Add a batch dimension

### 代码小知识：

1. **`T.Normalize` 里的这串奇怪数字是什么？**
`mean=[0.485, 0.456, 0.406]` 和 `std=[0.229, 0.224, 0.225]` 是计算机视觉领域非常著名的“ImageNet 标配参数”。它们对应着上百万张 ImageNet 图片在 RGB 三个通道上的平均值和标准差。在加载任何标准的预训练模型（如 ResNet、VGG 等）时，输入图片都必须用这组数字做标准化，否则模型的识别准确率会暴跌。
2. **最后的 `.unsqueeze(0)` 是干嘛的？**
单张图片处理完后的形状是 `(通道数, 高, 宽)`（即 `(3, 224, 224)`）。但是 PyTorch 的模型默认一次要接收一个“批次（Batch）”的图片。`.unsqueeze(0)` 的作用是在最前面强行塞入一个维度，把形状变成 `(1, 3, 224, 224)`，告诉模型：“这是一个大小为 1 的批次，里面有一张图片”。

In [5]:
def classify_image_with_resnet(image_url, topk=5):
    """
    Download an image, preprocess it, run it through
    ResNet-50, and print the top-k predictions.
    """
    
    # # Download and preprocess
    pil_img = download_image(image_url)
    img_t = preprocess_image(pil_img)
    
    # # Load a pretrained ResNet-50 model
    model = models.resnet50(pretrained=True)
    model.eval()  # Set model to evaluation mode
    
    # # Disable gradient calculation for speed and memory
    with torch.no_grad():
        output = model(img_t)
        
    # # Get label names and calculate probabilities
    labels = load_imagenet_labels()
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    
    # # Get top-k probabilities and corresponding indices
    topk_probs, topk_indices = torch.topk(probabilities, topk)
    
    # # Print the results
    print(f"\nImage URL: {image_url}")
    print("Top Predictions:")
    for i in range(topk):
        class_idx = topk_indices[i].item()
        prob = topk_probs[i].item()
        print(f"  {labels[class_idx]:<30} ({prob:.4f})")

### 核心硬核步骤拆解：

1. **`model.eval()` 必须加：**
加载完预训练模型后，必须调用 `.eval()`（评估模式）。因为 ResNet 内部含有 `BatchNorm`（批归一化）层，在训练和推理时的行为完全不同。如果不加这一行，直接拿去预测，模型输出的结果通常会是一团乱码。
2. **`with torch.no_grad():` 内存救星：**
默认情况下，PyTorch 会偷偷在后台帮你构建“计算图”以备后续计算梯度（用于反向传播训练）。但现在我们只是在做**推理/预测**，加了 `torch.no_grad()` 就可以强行关闭这个记录功能。它能让你的代码运行速度大幅提升，同时**砍掉大半的显存/内存占用**。
3. **`torch.nn.functional.softmax` 变成概率：**
模型最原始的输出（`output`）是一堆没有边界的正负数值（叫做 Logits）。用 `softmax` 函数处理后，这些数字就会被归一化到 `0` 到 `1` 之间，且所有类别的概率加起来刚好等于 `1`（即变成百分比）。
4. **`torch.topk` 捞出前五名：**
ImageNet 有 1000 个分类，我们往往只关心概率最高的那几个。`torch.topk(..., topk)` 会自动帮你排序，并把前 `k` 个最大的**概率值**以及对应的类别索引（ID）一起吐出来。

In [6]:
if __name__ == "__main__":
    # Replace this URL with any image URL you want to test
    test_image_url = (
        "https://images.pexels.com/photos/47547/"
        "squirrel-animal-cute-rodents-47547.jpeg?auto=compress&cs=tinysrgb&w"
        "=1260&h=750&dpr=2"
    )
    
    # Classify the image and print top-5 predictions
    classify_image_with_resnet(test_image_url, topk=5)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get th

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/voyagingpointer/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100.0%
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



Image URL: https://images.pexels.com/photos/47547/squirrel-animal-cute-rodents-47547.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=2
Top Predictions:
  fox squirrel                   (0.9990)
  hare                           (0.0003)
  marmot                         (0.0003)
  mink                           (0.0001)
  hamster                        (0.0001)
